&nbsp;

# [`fanta.bio`](https://fanta.bio/) Pre-Process

- `BED` ファイルの加工などを行う

In [1]:
import os
import datetime
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import matplotlib.pyplot as plt

from IPython.display import display


os.environ["POLARS_MAX_THREADS"] = "16"

In [2]:
VERSION = datetime.datetime.now().strftime("%y%m")
Execute_date = datetime.datetime.now().strftime("%y%m%d")
print(f"Execute Date: {Execute_date}")

Execute Date: 261001


In [3]:
# https://data.fanta.bio/cre/v1.2.1/00README_v1.2.1.txt
fantabio_bed = pl.read_csv(
    "../../Data/Database/fantabio/human-CREv1.2.1.hg38.cre-peaks.bed",
    separator="\t",
    has_header=False,
    new_columns=[
        "cre_chrom",
        "cre_chrom_start",
        "cre_chrom_end",
        "cre_id",
        "score",
        "strand",
        "thickStart",
        "thickEnd",
        "itemRgb",
        "cre_name",
        "pooled_cre_scores"
    ]
).with_columns(
    pl.col("pooled_cre_scores").str.extract(r"directionality:([^|]+)", 1).cast(pl.Float64, strict=False).alias("directionality"),
    pl.col("pooled_cre_scores").str.extract(r"class:([^|]+)", 1).alias("fantabio_cre_class"),
)

display(fantabio_bed)

cre_chrom,cre_chrom_start,cre_chrom_end,cre_id,score,strand,thickStart,thickEnd,itemRgb,cre_name,pooled_cre_scores,directionality,fantabio_cre_class
str,i64,i64,str,i64,str,i64,i64,str,str,str,f64,str
"""chr1""",10609,10687,"""FCHS_1""",0,""".""",10619,10687,"""0,0,255""","""cp1@DDX11L1""","""directionality:-1|class:PLA""",-1.0,"""PLA"""
"""chr1""",29398,29439,"""FCHS_2""",0,""".""",29398,29432,"""255,202,74""","""cp1@WASH7P""","""directionality:1|class:ELA""",1.0,"""ELA"""
"""chr1""",181113,181501,"""FCHS_447416""",0,""".""",181165,181342,"""255,202,74""","""cp1@DDX11L17""","""directionality:-0.684211|class…",-0.684211,"""ELA"""
"""chr1""",190831,191016,"""FCHS_447417""",0,""".""",190900,190947,"""255,202,74""","""FCHS_447417""","""directionality:0.485714|class:…",0.485714,"""ELA"""
"""chr1""",588413,588453,"""FCHS_3""",0,""".""",588423,588453,"""255,202,74""","""cp2@LOC101928626""","""directionality:-1|class:ELA""",-1.0,"""ELA"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrY""",56846801,56846977,"""FCHS_447311""",0,""".""",56846801,56846833,"""255,202,74""","""FCHS_447311""","""directionality:1|class:ELA""",1.0,"""ELA"""
"""chrY""",56855787,56855988,"""FCHS_447312""",0,""".""",56855844,56855977,"""255,202,74""","""FCHS_447312""","""directionality:0|class:ELA""",0.0,"""ELA"""
"""chrY""",56863206,56863354,"""FCHS_447313""",0,""".""",56863303,56863354,"""255,202,74""","""FCHS_447313""","""directionality:-1|class:ELA""",-1.0,"""ELA"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Only `PLA (Promoter Level Activity)` CRE BED file

- ChIP-Atlas の `Enrichment Analysis` 用に分離しておく
- GENCODE で定義されている transcript が近傍にないものであっても選抜している

In [4]:
fantabio_bed_pla = fantabio_bed.filter(
    pl.col("fantabio_cre_class") == "PLA"
)

fantabio_bed_pla.select(
    "cre_chrom",
    "cre_chrom_start",
    "cre_chrom_end"
).write_csv(
    "../../out/fantabio/BEDfile_ref/human-CREv1.2.1.hg38.cre-peaks_pla_all.bed",
    separator="\t",
    include_header=False
)

display(fantabio_bed_pla)

cre_chrom,cre_chrom_start,cre_chrom_end,cre_id,score,strand,thickStart,thickEnd,itemRgb,cre_name,pooled_cre_scores,directionality,fantabio_cre_class
str,i64,i64,str,i64,str,i64,i64,str,str,str,f64,str
"""chr1""",10609,10687,"""FCHS_1""",0,""".""",10619,10687,"""0,0,255""","""cp1@DDX11L1""","""directionality:-1|class:PLA""",-1.0,"""PLA"""
"""chr1""",629469,630126,"""FCHS_6""",0,""".""",629469,629922,"""204,0,0""","""cp1@MTND2P28""","""directionality:0.999887|class:…",0.999887,"""PLA"""
"""chr1""",629938,630317,"""FCHS_7""",0,""".""",629955,630144,"""153,0,0""","""cp2@MTND2P28""","""directionality:0.771711|class:…",0.771711,"""PLA"""
"""chr1""",631390,631686,"""FCHS_447419""",0,""".""",631400,631681,"""0,0,153""","""cp3@LOC101928626""","""directionality:-0.654321|class…",-0.654321,"""PLA"""
"""chr1""",633105,633714,"""FCHS_13""",0,""".""",633106,633542,"""204,0,0""","""cp1@MTATP6P1,cp1@MTATP8P1""","""directionality:0.999463|class:…",0.999463,"""PLA"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrY""",21177265,21177875,"""FCHS_593929""",0,""".""",21177429,21177875,"""0,0,204""","""FCHS_593929""","""directionality:-0.90566|class:…",-0.90566,"""PLA"""
"""chrY""",21254641,21254810,"""FCHS_447297""",0,""".""",21254641,21254741,"""255,0,0""","""FCHS_447297""","""directionality:1|class:PLA""",1.0,"""PLA"""
"""chrY""",21404652,21404951,"""FCHS_593934""",0,""".""",21404746,21404892,"""0,0,204""","""FCHS_593934""","""directionality:-0.943641|class…",-0.943641,"""PLA"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Only `ELA (Enhancer-Level Activity)` CRE BED file

- ChIP-Atlas の `Enrichment Analysis` 用に分離しておく
- GENCODE で定義されている transcript が近傍にないものであっても選抜している

In [5]:
fantabio_bed_ela = fantabio_bed.filter(
    pl.col("fantabio_cre_class") == "ELA"
)

fantabio_bed_ela.select(
    "cre_chrom",
    "cre_chrom_start",
    "cre_chrom_end"
).write_csv(
    "../../out/fantabio/BEDfile_ref/human-CREv1.2.1.hg38.cre-peaks_ela_all.bed",
    separator="\t",
    include_header=False
)

display(fantabio_bed_ela)

cre_chrom,cre_chrom_start,cre_chrom_end,cre_id,score,strand,thickStart,thickEnd,itemRgb,cre_name,pooled_cre_scores,directionality,fantabio_cre_class
str,i64,i64,str,i64,str,i64,i64,str,str,str,f64,str
"""chr1""",29398,29439,"""FCHS_2""",0,""".""",29398,29432,"""255,202,74""","""cp1@WASH7P""","""directionality:1|class:ELA""",1.0,"""ELA"""
"""chr1""",181113,181501,"""FCHS_447416""",0,""".""",181165,181342,"""255,202,74""","""cp1@DDX11L17""","""directionality:-0.684211|class…",-0.684211,"""ELA"""
"""chr1""",190831,191016,"""FCHS_447417""",0,""".""",190900,190947,"""255,202,74""","""FCHS_447417""","""directionality:0.485714|class:…",0.485714,"""ELA"""
"""chr1""",588413,588453,"""FCHS_3""",0,""".""",588423,588453,"""255,202,74""","""cp2@LOC101928626""","""directionality:-1|class:ELA""",-1.0,"""ELA"""
"""chr1""",605408,605588,"""FCHS_4""",0,""".""",605433,605562,"""255,202,74""","""FCHS_4""","""directionality:0.575758|class:…",0.575758,"""ELA"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrY""",56833095,56833397,"""FCHS_593939""",0,""".""",56833095,56833224,"""255,202,74""","""FCHS_593939""","""directionality:1|class:ELA""",1.0,"""ELA"""
"""chrY""",56846801,56846977,"""FCHS_447311""",0,""".""",56846801,56846833,"""255,202,74""","""FCHS_447311""","""directionality:1|class:ELA""",1.0,"""ELA"""
"""chrY""",56855787,56855988,"""FCHS_447312""",0,""".""",56855844,56855977,"""255,202,74""","""FCHS_447312""","""directionality:0|class:ELA""",0.0,"""ELA"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Only `PLA` and `ELA` CRE BED file (GENCODE-transcript-linked CREs)

- GENCODE version 49 の transcript が近傍にあるシスエレメントのみを選抜し、ChIP-Atlas Enrichment Analysis における background data とする

In [6]:
# Download from https://data.fanta.bio/cre/v1.2.0/human/ in 2026/03/10
fantabio_human_metadata = pl.read_ndjson(
    "../../Data/Database/fantabio/human-CREv1.2.1.hg38.cre-peaks_annotation.jsonl.gz"
).drop(
    "cre_old_names",
    "tss_ncbi_geneids",
    "tss_hgnc_mgi_ids",
    "tss_uniprot_ids",
    "tss_gene_names",
    "tss_gene_symbols",
    "tss_gene_synonyms",
    "tss_gene_sources"
).with_columns(
    # Q-score は後々フィルタリングの条件に使うかもしれないため、cast(pl.Int64) で整数型に変換しておく
    # print(fanta_bio_human_metadata["chip_atlas_antigens"].dtype) で確認できる
    pl.col("chip_atlas_antigens").list.eval(
        pl.struct([
            pl.element().struct.field("antigen").alias("antigen"),
            pl.element().struct.field("maxqscore").cast(pl.Int64).alias("maxqscore"),
            pl.element().struct.field("experiments").list.eval(
                pl.struct([
                    pl.element().struct.field("id").alias("id"),
                    pl.element().struct.field("qscore").cast(pl.Int64).alias("qscore"),
                ])
            ).alias("experiments"),
        ])
    ).alias("chip_atlas_antigens"),
).explode(
    "tss_transcripts"
).unnest(
    "tss_transcripts"
).filter(
    (pl.col("transcript_id").str.contains("ENST")) # Point1: Ensembl Transcript ID のみに絞る
).with_columns(
    # Point2: "transcript_distance" は str 型で読み込まれるが, int 型に変換しておく
    pl.col("transcript_distance")
    .cast(pl.Float64, strict=False)
    .round(0) # 小数点以下を四捨五入 (8件のみ .5 が存在)
    .cast(pl.Int64)
)

pl.Config.set_tbl_rows(4)
print(fantabio_human_metadata.select("cre_id").n_unique()) # Ensembl Transcript ID のみに絞った場合のユニークな CRE ID 数
display(fantabio_human_metadata)

162925


cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_id,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers
str,str,i64,str,str,list[str],str,str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]]
"""FCHS_1""","""chr1""",10610,"""10687""","""cp1@DDX11L1""","[""ENST00000832824.1""]","""434""","""ENST00000832824.1""",434,"[{""EH38E2776517"",""pELS""}]","[{""EGR1"",2552,[{""SRX22828712"",2552}, {""SRX22828708"",1544}, {""SRX22828709"",1187}]}, {""BRD3"",1874,[{""SRX366792"",1874}, {""SRX366793"",621}, {""SRX366794"",537}]}, … {""NRF1"",807,[{""SRX2636389"",807}, {""SRX23881992"",770}, {""SRX1280449"",580}]}]",null,null,null
"""FCHS_1""","""chr1""",10610,"""10687""","""cp1@DDX11L1""","[""ENST00000832824.1""]","""434""","""ENST00000832825.1""",438,"[{""EH38E2776517"",""pELS""}]","[{""EGR1"",2552,[{""SRX22828712"",2552}, {""SRX22828708"",1544}, {""SRX22828709"",1187}]}, {""BRD3"",1874,[{""SRX366792"",1874}, {""SRX366793"",621}, {""SRX366794"",537}]}, … {""NRF1"",807,[{""SRX2636389"",807}, {""SRX23881992"",770}, {""SRX1280449"",580}]}]",null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_593938""","""chrY""",26409259,"""26409692""","""FCHS_593938""","[""ENST00000725555.1""]","""-6""","""ENST00000344424.4""",123,"[{""EH38E3951258"",""pELS""}, {""EH38E4535230"",""pELS""}]","[{""MYCN"",537,[{""SRX6935376"",537}]}]",null,null,null
"""FCHS_593940""","""chrY""",56881615,"""56881922""","""FCHS_593940""","[""ENST00000810723.1""]","""0""","""ENST00000810723.1""",0,null,"[{""RELA"",1452,[{""SRX5938177"",1452}]}, {""RNF2"",939,[{""SRX18954702"",939}, {""SRX18954704"",699}]}, … {""DLX5"",504,[{""SRX17306262"",504}]}]",null,"[{""hg19::chrY:59027988..59028000,+;hg_201233.1"",""hg_201233.1""}]",null


In [7]:
fantabio_human_metadata_viz = fantabio_human_metadata.join(
    fantabio_bed.select("cre_id", "cre_name", "fantabio_cre_class"),
    on="cre_id",
    how="inner"
).with_columns(
    pl.col("transcript_distance").abs().alias("transcript_distance_abs") # Point: 絶対値に変換
).select(
    "cre_id",
    "transcript_id",
    "transcript_distance",
    "transcript_distance_abs",
    "fantabio_cre_class",
    "cre_name"
)

display(fantabio_human_metadata_viz)

cre_id,transcript_id,transcript_distance,transcript_distance_abs,fantabio_cre_class,cre_name
str,str,i64,i64,str,str
"""FCHS_1""","""ENST00000832824.1""",434,434,"""PLA""","""cp1@DDX11L1"""
"""FCHS_1""","""ENST00000832825.1""",438,438,"""PLA""","""cp1@DDX11L1"""
…,…,…,…,…,…
"""FCHS_593938""","""ENST00000344424.4""",123,123,"""ELA""","""FCHS_593938"""
"""FCHS_593940""","""ENST00000810723.1""",0,0,"""PLA""","""FCHS_593940"""


In [8]:
plot_df = fantabio_human_metadata_viz.to_pandas()

# 5 bins: 0-100, 101-200, ..., 401-500
bin_edges = [0, 101, 201, 301, 401, 501]
bin_labels = ["0-100", "101-200", "201-300", "301-400", "401-500"]

plot_df["distance_bin"] = pd.cut(
    plot_df["transcript_distance_abs"],
    bins=bin_edges,
    labels=bin_labels,
    right=False,
    include_lowest=True,
)

count_df = (
    plot_df.groupby(["distance_bin", "fantabio_cre_class"], observed=True)
    .size()
    .to_frame("count")
    .reset_index()
)

pivot_df = (
    count_df.pivot(index="distance_bin", columns="fantabio_cre_class", values="count")
    .fillna(0)
    .astype(int)
    .reindex(bin_labels)
)

classes = pivot_df.columns.tolist()
palette = sns.color_palette("Set2", n_colors=len(classes))
color_map = dict(zip(classes, palette))

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(16, 10))

bottom = np.zeros(len(pivot_df))
x = np.arange(len(pivot_df.index))

for cls in classes:
    heights = pivot_df[cls].to_numpy()
    ax.bar(
        x,
        heights,
        bottom=bottom,
        label=cls,
        color=color_map[cls],
        edgecolor="white",
        linewidth=0.5,
    )
    bottom += heights

ax.set_xticks(x)
ax.set_xticklabels(pivot_df.index, fontsize=14)
ax.set_xlabel("|transcript distance| bin (bp)", fontsize=18)
ax.set_ylabel("Count (lncRNA–CRE pairs)", fontsize=18)
ax.set_title("100 bp bins by GENCODE v49 transcript - CRE pairs", fontsize=18)
ax.legend(title="fantabio CRE class", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.tick_params(axis="y", labelsize=14)

plt.tight_layout()
plt.show()

<Figure size 1600x1000 with 1 Axes>

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Only `PLA (Promoter-Level Activity)` CRE BED file with GENCODE transcript

- Promoter Level Activity (PLA) CREs のうち、GENCODE-transcript-linked CREs を抽出する

In [9]:
fantabio_bed_pla_gencode_transcript_linked = fantabio_bed.filter(
    pl.col("fantabio_cre_class") == "PLA"
).join(
    fantabio_human_metadata.select("cre_id").unique(),
    on="cre_id",
    how="semi",
)

fantabio_bed_pla_gencode_transcript_linked.select(
    "cre_chrom",
    "cre_chrom_start",
    "cre_chrom_end"
).write_csv(
    "../../out/fantabio/BEDfile_ref/human-CREv1.2.1.hg38.cre-peaks_pla_gencode_transcript_linked.bed",
    separator="\t",
    include_header=False
)

print(fantabio_bed_pla.height)
print(
    fantabio_bed_pla.filter(
        ~pl.col("cre_id").is_in(fantabio_human_metadata["cre_id"]) # 近傍に GENCODE-transcript が存在しない CRE をカウント
    ).height
)

display(fantabio_bed_pla_gencode_transcript_linked)

72627


/tmp/ipykernel_75317/1286234112.py:21: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  fantabio_bed_pla.filter(


20789


cre_chrom,cre_chrom_start,cre_chrom_end,cre_id,score,strand,thickStart,thickEnd,itemRgb,cre_name,pooled_cre_scores,directionality,fantabio_cre_class
str,i64,i64,str,i64,str,i64,i64,str,str,str,f64,str
"""chr1""",10609,10687,"""FCHS_1""",0,""".""",10619,10687,"""0,0,255""","""cp1@DDX11L1""","""directionality:-1|class:PLA""",-1.0,"""PLA"""
"""chr1""",629469,630126,"""FCHS_6""",0,""".""",629469,629922,"""204,0,0""","""cp1@MTND2P28""","""directionality:0.999887|class:…",0.999887,"""PLA"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrY""",26408845,26409331,"""FCHS_593937""",0,""".""",26409122,26409180,"""153,0,0""","""FCHS_593937""","""directionality:0.638596|class:…",0.638596,"""PLA"""
"""chrY""",56881614,56881922,"""FCHS_593940""",0,""".""",56881693,56881854,"""204,0,0""","""FCHS_593940""","""directionality:0.989305|class:…",0.989305,"""PLA"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Only `ELA (Enhancer-Level Activity)` CRE BED file with GENCODE transcript

- Enhancer Level Activity (ELA) CREs のうち、GENCODE-transcript-linked CREs を抽出する

In [10]:
fantabio_bed_ela_gencode_transcript_linked = fantabio_bed.filter(
    pl.col("fantabio_cre_class") == "ELA"
).join(
    fantabio_human_metadata.select("cre_id").unique(),
    on="cre_id",
    how="semi",
)

fantabio_bed_ela_gencode_transcript_linked.select(
    "cre_chrom",
    "cre_chrom_start",
    "cre_chrom_end"
).write_csv(
    "../../out/fantabio/BEDfile_ref/human-CREv1.2.1.hg38.cre-peaks_ela_gencode_transcript_linked.bed",
    separator="\t",
    include_header=False
)


print(fantabio_bed_ela.height)
print(
    fantabio_bed_ela.filter(
        ~pl.col("cre_id").is_in(fantabio_human_metadata["cre_id"]) # 近傍に GENCODE-transcript が存在しない CRE をカウント
    ).height
)

display(fantabio_bed_ela_gencode_transcript_linked)

441268


/tmp/ipykernel_75317/2787901369.py:22: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  fantabio_bed_ela.filter(


330181


cre_chrom,cre_chrom_start,cre_chrom_end,cre_id,score,strand,thickStart,thickEnd,itemRgb,cre_name,pooled_cre_scores,directionality,fantabio_cre_class
str,i64,i64,str,i64,str,i64,i64,str,str,str,f64,str
"""chr1""",29398,29439,"""FCHS_2""",0,""".""",29398,29432,"""255,202,74""","""cp1@WASH7P""","""directionality:1|class:ELA""",1.0,"""ELA"""
"""chr1""",181113,181501,"""FCHS_447416""",0,""".""",181165,181342,"""255,202,74""","""cp1@DDX11L17""","""directionality:-0.684211|class…",-0.684211,"""ELA"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrY""",26409025,26409052,"""FCHS_447307""",0,""".""",26409025,26409047,"""255,202,74""","""FCHS_447307""","""directionality:1|class:ELA""",1.0,"""ELA"""
"""chrY""",26409258,26409692,"""FCHS_593938""",0,""".""",26409531,26409572,"""255,202,74""","""FCHS_593938""","""directionality:0.666667|class:…",0.666667,"""ELA"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

In [11]:
# Inspect most frequent itemRgb values
# BED file に記載がある itemRgb の値を確認し、その値をもとにわかりやすいように色名をアサインしておく
# fantabio_bed_color = fantabio_bed.select("itemRgb").unique()

# pl.Config.set_tbl_rows(12)
# display(fantabio_bed_color)

In [12]:
# Map exact RGB strings to human-readable names
rgb_to_name = {
    "153,0,0": "dark_red",
    "101,0,0": "dark_red",
    "50,0,0": "dark_red",
    "204,0,0": "red",
    "255,0,0": "red",
    "0,0,51": "dark_blue",
    "0,0,102": "dark_blue",
    "0,0,153": "blue",
    "0,0,204": "blue",
    "0,0,255": "blue",
    "0,0,0": "black",
    "255,202,74": "yellow",
}

fantabio_bed_select = fantabio_bed.with_columns(
    pl.col("itemRgb").replace_strict(
        rgb_to_name,
        default="other"
    ).alias("itemRgb_name")
).select(
    "cre_id",
    "cre_name",
    "directionality",
    "fantabio_cre_class",
    "itemRgb",
    "itemRgb_name"
)

display(fantabio_bed_select)

cre_id,cre_name,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,str,f64,str,str,str
"""FCHS_1""","""cp1@DDX11L1""",-1.0,"""PLA""","""0,0,255""","""blue"""
"""FCHS_2""","""cp1@WASH7P""",1.0,"""ELA""","""255,202,74""","""yellow"""
…,…,…,…,…,…
"""FCHS_447314""","""FCHS_447314""",-1.0,"""ELA""","""255,202,74""","""yellow"""
"""FCHS_593940""","""FCHS_593940""",0.989305,"""PLA""","""204,0,0""","""red"""
